# Couplers — acoplamiento excesivo

Delatan cómo las clases interactúan entre sí: demasiado acoplamiento o delegación rota.

## Serie: Refactorización y Code Smells

Este contenido está dividido en 6 notebooks — uno por categoría de code
smell (clasificación de refactoring.guru) más un cierre de ejercicios:

1. `01_bloaters.ipynb` — Bloaters
2. `02_object_orientation_abusers.ipynb` — Object-Orientation Abusers
3. `03_change_preventers.ipynb` — Change Preventers
4. `04_dispensables.ipynb` — Dispensables
5. **`05_couplers.ipynb`** — Couplers
6. `06_ejercicios_autoevaluacion.ipynb` — Ejercicios y autoevaluación


### 1. Feature Envy (envidia de características)

**Definición:** Un método de una clase está más interesado en los datos de otra clase que en los de la suya propia.

**Síntoma:** El método recorre y opera directamente sobre los atributos internos de un objeto ajeno.

**Técnica de refactor:** Move Method

#### Con el smell

In [ ]:
class Carrito:
    def __init__(self, items):
        self.items = items  # lista de (precio, cantidad)

class Factura:
    def calcular_total(self, carrito):
        total = 0
        for precio, cantidad in carrito.items:
            total += precio * cantidad
        return total  # envidia los datos de Carrito

carrito = Carrito([(80, 1), (30, 2)])
print(Factura().calcular_total(carrito))

#### Refactorizado

In [ ]:
class Carrito:
    def __init__(self, items):
        self.items = items

    def calcular_total(self):
        return sum(precio * cantidad for precio, cantidad in self.items)

class Factura:
    def calcular_total(self, carrito):
        return carrito.calcular_total()

carrito = Carrito([(80, 1), (30, 2)])
print(Factura().calcular_total(carrito))

**Explicación:** `Move Method` traslada el cálculo a `Carrito`, la clase dueña de los datos. `Factura` ahora solo pide el resultado en vez de recorrer datos ajenos.

### 2. Message Chains (cadenas de mensajes)

**Definición:** Una secuencia de llamadas encadenadas del tipo `a.getB().getC().getD()` que expone toda la estructura interna de una jerarquía de objetos.

**Síntoma:** El código cliente necesita conocer Cliente → Dirección → Ciudad solo para leer un nombre.

**Técnica de refactor:** Hide Delegate

#### Con el smell

In [ ]:
class Ciudad:
    def __init__(self, nombre):
        self.nombre = nombre

class Direccion:
    def __init__(self, ciudad):
        self.ciudad = ciudad

class Cliente:
    def __init__(self, direccion):
        self.direccion = direccion

cliente = Cliente(Direccion(Ciudad("Bogotá")))
nombre_ciudad = cliente.direccion.ciudad.nombre  # cadena de mensajes
print(nombre_ciudad)

#### Refactorizado

In [ ]:
class Ciudad:
    def __init__(self, nombre):
        self.nombre = nombre

class Direccion:
    def __init__(self, ciudad):
        self.ciudad = ciudad

class Cliente:
    def __init__(self, direccion):
        self.direccion = direccion

    def ciudad(self):  # oculta la cadena interna
        return self.direccion.ciudad.nombre

cliente = Cliente(Direccion(Ciudad("Bogotá")))
print(cliente.ciudad())

**Explicación:** `Hide Delegate` agrega un método `ciudad()` en `Cliente` que oculta la cadena. El código cliente ya no necesita saber que existen `Direccion` y `Ciudad` como pasos intermedios.

### 3. Inappropriate Intimacy (intimidad inapropiada)

**Definición:** Dos clases dependen de forma tan estrecha que una manipula directamente los detalles internos ("privados") de la otra.

**Síntoma:** Una clase modifica atributos con guion bajo (`_saldo`) de otra clase en lugar de pedirle que lo haga ella misma.

**Técnica de refactor:** Move Method / encapsular el estado

#### Con el smell

In [ ]:
class CuentaBancaria:
    def __init__(self, saldo):
        self._saldo = saldo  # "privado" por convención

class Banco:
    def transferir(self, origen, destino, monto):
        # accede directo al interno de la cuenta
        origen._saldo -= monto
        destino._saldo += monto

a = CuentaBancaria(1000)
b = CuentaBancaria(0)
Banco().transferir(a, b, 300)
print(a._saldo, b._saldo)

#### Refactorizado

In [ ]:
class CuentaBancaria:
    def __init__(self, saldo):
        self._saldo = saldo

    def retirar(self, monto):
        self._saldo -= monto

    def depositar(self, monto):
        self._saldo += monto

    def saldo(self):
        return self._saldo

class Banco:
    def transferir(self, origen, destino, monto):
        origen.retirar(monto)
        destino.depositar(monto)

a = CuentaBancaria(1000)
b = CuentaBancaria(0)
Banco().transferir(a, b, 300)
print(a.saldo(), b.saldo())

**Explicación:** `CuentaBancaria` ahora protege su propio estado a través de `retirar()` y `depositar()`. `Banco` ya no conoce ni toca el atributo interno `_saldo` de otra clase: la intimidad inapropiada desaparece.

### 4. Middle Man (intermediario)

**Definición:** Una clase cuyos métodos solo delegan el trabajo a otro objeto, sin agregar ningún comportamiento propio.

**Síntoma:** La mayoría de los métodos de una clase son una sola línea que llama al método equivalente de otro objeto que tiene como atributo.

**Técnica de refactor:** Remove Middle Man

#### Con el smell

In [ ]:
class Motor:
    def encender(self):
        print("Motor encendido")

    def apagar(self):
        print("Motor apagado")

class Auto:
    def __init__(self, motor):
        self._motor = motor

    def encender(self):
        return self._motor.encender()

    def apagar(self):
        return self._motor.apagar()

Auto(Motor()).encender()

#### Refactorizado

In [ ]:
class Motor:
    def encender(self):
        print("Motor encendido")

    def apagar(self):
        print("Motor apagado")

class Auto:
    def __init__(self, motor):
        self.motor = motor  # acceso directo, sin envoltorio

Auto(Motor()).motor.encender()

**Explicación:** `Remove Middle Man` elimina los métodos de `Auto` que solo reenviaban la llamada a `Motor`, y expone el objeto directamente. El cliente pierde una capa que no agregaba ningún valor.

### 5. Incomplete Library Class (clase de librería incompleta)

**Definición:** Necesitas un método adicional en una clase de una librería externa (o de la librería estándar), pero no puedes modificarla directamente.

**Síntoma:** El código reimplementa manualmente, cada vez que hace falta, una operación que "debería" ser un método de una clase que no es tuya.

**Técnica de refactor:** Introduce Foreign Method

#### Con el smell

In [ ]:
from datetime import date

hoy = date(2026, 1, 1)
festivos = {date(2026, 1, 1), date(2026, 12, 25)}

# cada vez que hace falta, se repite la misma comprobación manual
if hoy in festivos:
    print("Hoy es festivo")

#### Refactorizado

In [ ]:
from datetime import date

FESTIVOS = {date(2026, 1, 1), date(2026, 12, 25)}

def es_festivo(fecha):  # método externo para `date`
    return fecha in FESTIVOS

hoy = date(2026, 1, 1)
if es_festivo(hoy):
    print("Hoy es festivo")

**Explicación:** `Introduce Foreign Method` crea una función externa (`es_festivo`) que actúa como si fuera un método de `date`, sin modificar la clase de la librería estándar. La comprobación queda centralizada y con un nombre claro, en vez de repetirse cada vez que se necesita.